# 4d. Net and gross revenue retention

**What an investor reads into this.** NRR answers "if Northbridge stopped selling to new customers today, how much would last year's customer base be worth in a year?" Above 100% the existing base grows on its own; below 100% the business has to win new customers just to stand still. GRR removes expansion and shows the pure loss rate (churn and downgrades), which is the floor under the revenue a buyer is paying for. Both are read alongside price increases: a price rise lifts NRR without any change in customer behaviour.

**Limit:** NRR and GRR can only be computed for 2023, 2024 and 2025. The data starts in January 2022, so there is no December 2021 opening point.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nb_utils import connect, q, sql_file, save_table, pct
import charts as ch

pd.set_option("display.width", 220)
con = connect()
dec = sql_file(con, "dec_mrr_customer_line").groupby(["customer_id", "year"], as_index=False).dec_mrr.sum()
k = sql_file(con, "uplift_factors").set_index("year").k
years = [y for y in sorted(dec.year.unique()) if y - 1 in dec.year.values]

def nrr_grr(y, exclude=(), deflate=False):
    o = dec[(dec.year == y - 1) & ~dec.customer_id.isin(exclude)].set_index("customer_id").dec_mrr
    c = dec[dec.year == y].set_index("customer_id").dec_mrr.reindex(o.index, fill_value=0.0)
    if deflate:
        c = c / float(k[y])
    return {"opening_mrr": o.sum(), "closing_mrr_same_customers": c.sum(), "closing_mrr_capped": np.minimum(c, o).sum(),
            "nrr_pct": 100 * c.sum() / o.sum(), "grr_pct": 100 * np.minimum(c, o).sum() / o.sum(),
            "customers_opening": len(o), "customers_retained": int((c > 0).sum())}

## 1. NRR and GRR, with sensitivities

- **Excluding the top 5:** the five customers with the largest opening-December MRR are removed from both numerator and denominator.
- **Excluding the price increase:** closing MRR is divided by the year's uniform uplift factor (1.00, 1.05, 1.07; see `docs/metric_definitions.md`).

In [2]:
rows = []
for y in years:
    top5 = dec[dec.year == y - 1].sort_values(["dec_mrr", "customer_id"], ascending=[False, True]).head(5).customer_id.tolist()
    for name, kw in [("reported", {}), ("excluding top 5 customers", {"exclude": top5}),
                     ("excluding price increase", {"deflate": True}),
                     ("excluding top 5 and price increase", {"exclude": top5, "deflate": True})]:
        rows.append({"year": y, "basis": name, "uplift_factor": float(k[y]), **nrr_grr(y, **kw)})
nrr = save_table(pd.DataFrame(rows), "4d_nrr_grr")
nrr.pivot(index="basis", columns="year", values=["nrr_pct", "grr_pct"]).round(1)

nrr_pct             grr_pct            
year                                  2023  2024  2025    2023  2024  2025
basis                                                                     
excluding price increase              95.8  94.0  89.2    94.5  92.7  88.2
excluding top 5 and price increase    94.0  92.8  90.9    92.3  91.0  89.6
excluding top 5 customers             94.0  97.5  97.3    92.3  91.1  89.7
reported                              95.8  98.7  95.4    94.5  92.9  88.8

## 2. What good looks like

Indicative ranges for B2B SaaS selling to small and mid-sized businesses, from general industry knowledge. They are **indicative, not sourced**: no benchmark survey was read for this analysis, and published figures vary widely by customer size, contract length and pricing model.

| Measure | Weaker | Typical | Strong |
|---|---|---|---|
| GRR | below 80% | about 80% to 90% | above 90% |
| NRR | below 90% | about 90% to 105% | above 105% |

In [3]:
bench = save_table(pd.DataFrame([
    {"measure": "GRR", "weaker": "below 80%", "typical": "about 80% to 90%", "strong": "above 90%", "basis": "indicative, not sourced"},
    {"measure": "NRR", "weaker": "below 90%", "typical": "about 90% to 105%", "strong": "above 105%", "basis": "indicative, not sourced"},
]), "4d_benchmarks_indicative")

## 3. Chart

In [4]:
rep = nrr[nrr.basis == "reported"].set_index("year")
exp = nrr[nrr.basis == "excluding price increase"].set_index("year")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
for ax, col, title in zip(axes, ["nrr_pct", "grr_pct"], ["NRR", "GRR"]):
    x = np.arange(len(years)); w = 0.36
    b1 = ax.bar(x - w / 2, rep[col], w, color=ch.BLUE, label="Reported", edgecolor=ch.SURFACE, linewidth=1.5)
    b2 = ax.bar(x + w / 2, exp[col], w, color=ch.ORANGE, label="Excluding price increase", edgecolor=ch.SURFACE, linewidth=1.5)
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.6, f"{b.get_height():.1f}%", ha="center", fontsize=8, color=ch.TEXT)
    ax.axhline(100, color=ch.TEXT_2, lw=1, ls=(0, (3, 3)))
    ax.set_xticks(x); ax.set_xticklabels([str(y) for y in years])
    ax.set_ylim(70, 110); ax.yaxis.set_major_formatter(ch.pct_fmt())
    ax.set_title(f"{title} by year (2023 to 2025 only)")
    ax.set_xlabel("Year (December to December)")
axes[0].set_ylabel("% of opening-December MRR")
axes[0].legend(loc="lower left")
print(ch.save(fig, "4d_nrr_grr"))

outputs/charts/4d_nrr_grr.png


In [5]:
nrr

,year,basis,uplift_factor,opening_mrr,closing_mrr_same_customers,closing_mrr_capped,nrr_pct,grr_pct,customers_opening,customers_retained
0,2023,reported,1.00,2417140.30,2.314809e+06,2.284726e+06,95.766411,94.521877,1010,920
1,2023,excluding top 5 customers,1.00,1711723.63,1.609392e+06,1.579310e+06,94.021711,92.264294,1005,915
2,2023,excluding price increase,1.00,2417140.30,2.314809e+06,2.284726e+06,95.766411,94.521877,1010,920
3,2023,excluding top 5 and price increase,1.00,1711723.63,1.609392e+06,1.579310e+06,94.021711,92.264294,1005,915
4,2024,reported,1.05,2731778.55,2.697551e+06,2.538154e+06,98.747049,92.912156,1205,1073
5,2024,excluding top 5 customers,1.05,2026361.88,1.975238e+06,1.845279e+06,97.477071,91.063663,1200,1068
6,2024,excluding price increase,1.05,2731778.55,2.569096e+06,2.531658e+06,94.044808,92.674359,1205,1073
7,2024,excluding top 5 and price increase,1.05,2026361.88,1.881179e+06,1.843742e+06,92.835306,90.987776,1200,1068
8,2025,reported,1.07,3124736.34,2.982357e+06,2.773799e+06,95.443477,88.769051,1343,1172
9,2025,excluding top 5 customers,1.07,2402423.85,2.337281e+06,2.153774e+06,97.288444,89.650034,1338,1167
